# Lp(a) Reddit Topic Modeling: JAHA revision analysis

Analysis for the revised manuscript. Written for Google Colab with a GPU runtime (an A100 was used); run top to bottom. Reported values are collected in the dict `R` and written to `revision_results.json` in section 14.

| Section | Analysis |
|---|---|
| 1 | Corpus audit: Lp(a)-explicit subset, promoted posts, 2015 |
| 2 | Keyword themes and precision sample |
| 3 | Nesting and repeat posters |
| 4 | Descriptive growth |
| 5 | Seeded primary topic model and number of groups |
| 6 | Tables and figures |
| 7 | Thematic group share over time, with sensitivity analyses |
| 8 | Topic coherence (NPMI) |
| 9 | Clinician label validation and scoring |
| 10 | Sensitivity, stability, and choice of k |
| 11 | Platform denominators (Arctic Shift) |
| 12 | Platform-adjusted negative binomial growth model |
| 13 | Full-corpus sensitivity run (optional) |
| 14 | Export of results |

## Setup

In [ ]:
!pip install bertopic
!pip install llama-cpp-python
!pip install datamapplot
!pip install langdetect
!pip install statsmodels
import os, re, sys, json, time, importlib, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, umap, hdbscan, bertopic
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_rand_score, silhouette_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Package import

In [ ]:
# Directory containing the lpa_topic_modeling package
V3_PATH = '/path/to/Lp-a-Reddit-Topic-Modeling'
if V3_PATH not in sys.path:
    sys.path.insert(0, V3_PATH)
import lpa_topic_modeling
importlib.reload(lpa_topic_modeling)
from lpa_topic_modeling import (TopicModeler, GroupLabeler, TopicClusterer,
                                fig1a_temporal, fig1b_clusters, config)
try:
    from lpa_topic_modeling import fig_silhouette_scan
except ImportError:
    fig_silhouette_scan = None
print("SEED =", config.SEED, "| N_GROUPS =", config.N_GROUPS, "| DEVICE =", config.DEVICE)

In [ ]:
# Override TopicClusterer._topic_embedding to exclude the outlier topic (-1) before grouping;
# the package method clusters every c-TF-IDF row, including -1.
from sklearn.preprocessing import normalize as _l2norm
from umap import UMAP as _UMAP
def _topic_embedding_no_outliers(self):
    info = self.modeler.topic_model.get_topic_info().sort_values("Topic").reset_index(drop=True)
    X = self.modeler.topic_model.c_tf_idf_.toarray()
    assert len(X) == len(info), "c-TF-IDF rows and topic_info rows differ"
    keep = (info["Topic"] != -1).values
    info, X = info[keep].reset_index(drop=True), _l2norm(X[keep], norm="l2", axis=1)
    emb = _UMAP(n_components=config.GROUP_UMAP_COMPONENTS, n_neighbors=min(config.GROUP_UMAP_NEIGHBORS, len(X) - 1),
                min_dist=config.GROUP_UMAP_MIN_DIST, metric=config.GROUP_UMAP_METRIC,
                random_state=self.seed).fit_transform(X)
    return info, emb
TopicClusterer._topic_embedding = _topic_embedding_no_outliers

## Paths and run flags
Set `V3_PATH` (Package import cell) and `BASE` to local or Drive folders before running. The optional name-mask file is read from `BASE`.

In [ ]:
BASE = '/path/to/project_dir'
DATA = f'{BASE}/filtered_df.json'
REV  = f'{BASE}/jaha_revision_dedup'; os.makedirs(REV, exist_ok=True)   # revision outputs
OUT  = REV                                   # tables and figures are written to OUT
META_P   = f'{REV}/metadata_primary.pkl'     # seeded primary model, after outlier reduction
MODEL_P  = f'{REV}/model_primary.bertopic'
GROUPS_P = f'{REV}/group_summaries_primary.json'
EMB_P    = f'{REV}/embeddings_primary.npy'

RESUME           = False   # True = reload the fitted model, groups and saved results (skips about 1 h of fitting);
                           # False = embed, run the clustering diagnostic, fit and label from scratch
RUN_LLM          = True    # Llama-3 topic and group labels; False = keyword-only run
RUN_SENSITIVITY  = True    # section 10 grid (roughly 30-60 min on A100)
RUN_DENOMINATORS = True    # section 11; requires internet access
RUN_FULL_CORPUS  = False   # section 13, optional
Y0, Y1 = 2011, 2024        # growth window; 2025 is a partial year
R = {}                     # collected results, exported in section 14

# Names of individuals are masked in all displayed labels, keyword lists, exemplars, rater documents and subreddit
# names; the underlying data and models are unchanged. Patterns are read from a private, git-ignored file.
# Format: {"person": {regex: replacement}, "subreddit": {name: replacement}}. Without the file, nothing is masked.
MASK_P = f'{BASE}/name_mask.json'
_mask = json.load(open(MASK_P)) if os.path.exists(MASK_P) else {"person": {}, "subreddit": {}}
if not _mask["person"]: print("WARNING: name_mask.json not found - labels and exemplars will not be masked")
PERSON_MASK, SUBREDDIT_MASK = _mask["person"], _mask["subreddit"]
def mask_names(x):
    x = str(x)
    for p, r in PERSON_MASK.items(): x = re.sub(p, r, x)
    return x
def show_sub(x): return SUBREDDIT_MASK.get(x, x)

# Adjudicated group names. Keys are the LLM-generated group labels; values are the names used in figures and tables.
NAME_MAP = {
    'Cardiovascular Disease Prevention and Treatment': 'Estimating and managing elevated Lp(a) risk',
    'Coronary Artery Calcium Scoring and Cardiovascular Risk Assessment': 'Coronary artery calcium scoring and cardiac symptoms',
    'Effects of dietary fat intake on insulin sensitivity and cardiovascular disease risk': 'Diet, saturated fat, and metabolic health',
    'Lipoprotein(a) and Cardiovascular Disease Risk': 'Evidence on atherogenic lipoproteins',
    'Lipid Profile Analysis': 'Sharing and interpreting lab results',
    'Genetic and Lifestyle Factors Contributing to High Risk of Heart Disease': 'Family history and high-risk prevention',
    'Management of High Risk Patients with LDL-C and Lp(a) Elevation': 'Non-statin and combination lipid-lowering therapy',
    'Cardiovascular Disease Risk Factors': 'ApoB targets',
    'Lipoprotein(a) Testing and Heart Disease Risk Assessment': 'Getting tested for Lp(a)',
}
# Adjudicated topic labels (Table S4), keyed by topic number; five topics share their LLM label with their group.
TOPIC_FINAL = {
    0: 'Coronary artery calcium scoring and cardiovascular risk',
    1: 'Managing elevated Lp(a): statins and niacin',
    2: 'Comprehensive blood panels and hormone testing',
    3: 'Dietary fat, insulin sensitivity, and cardiovascular risk',
    4: 'PCSK9 inhibitor and statin treatment experiences',
    5: 'Lp(a) as a cardiovascular risk factor',
    6: 'Lp(a) and cholesterol results in younger adults',
    7: 'Dietary strategies for lowering LDL cholesterol',
    8: 'Familial hypercholesterolemia, family history, and heart disease risk',
    9: 'ApoB-based assessment of atherogenic risk',
    10: 'Lp(a) testing and heart disease risk assessment',
    11: 'Lipid panel and Lp(a) test results',
    12: 'Cardiac risk factors and heart attack symptoms',
    13: 'Interpreting and managing Lp(a) results',
    14: 'Lp(a) in risk calculators and risk factors',
    15: 'Requesting Lp(a) testing',
    16: 'High Lp(a), aortic valve disease, and primary prevention',
    17: 'Insulin resistance, glucose, and lipid biomarkers',
    18: 'Evidence on LDL and atherosclerosis',
    19: 'ApoB testing and treatment targets',
    20: 'Dietary management of high LDL cholesterol and apoB',
    21: 'Dietary changes and cholesterol results',
    22: 'Ezetimibe and statin dosing for high Lp(a) and LDL cholesterol',
    23: 'CAC scores and cardiovascular risk',
    24: 'Advanced lipid testing, ordering, and cost',
    25: 'LDL cholesterol targets and early prevention',
}
def final_name(l): return NAME_MAP.get(l, l)
import matplotlib.text as _mtext
def relabel_figure(fig):
    # replace LLM labels with final names in all text objects of a figure (legends, annotations)
    for t in fig.findobj(_mtext.Text):
        s = t.get_text().strip()
        if s in NAME_MAP: t.set_text(NAME_MAP[s])


## 1. Corpus audit: Lp(a)-explicit subset, promoted posts, 2015 (R1.1, R1.2, R1.5, R3.1)
Removes duplicate rows, flags documents that explicitly name Lp(a), and flags promoted or recruitment content. The topic-model input is the Lp(a)-explicit, non-promoted subset.

In [ ]:
reddit_df = pd.read_json(DATA)
df = reddit_df.copy()
# The retrieved file contains repeated rows for the same Reddit item (one item can match several search terms).
# Keep one row per item, preferring a non-deleted author and the longest body (copies may precede or follow an edit).
n_rows_retrieved = len(df)
df["_del"] = df["author"].fillna("[deleted]").isin(["[deleted]", "[removed]"])
df["_len"] = df["body"].fillna("").str.len()
df = (df.sort_values(["_del", "_len"], ascending=[True, False])
        .drop_duplicates(["id", "type_of_entry"]).sort_index().drop(columns=["_del", "_len"]))
print(f"rows retrieved {n_rows_retrieved} | unique Reddit items {len(df)} | duplicate rows removed {n_rows_retrieved - len(df)}")
df["year"] = pd.to_datetime(pd.to_numeric(df["created_utc"], errors="coerce"), unit="s", utc=True).dt.year
txt = (df.get("title", "").fillna("").astype(str) + " " + df.get("selftext", "").fillna("").astype(str)
       + " " + df.get("body", "").fillna("").astype(str))
low = txt.str.lower()

# Word-bounded Lp(a) pattern. A prefix match on "lipoprotein a" would also capture phrases such as
# "high-density lipoprotein and", "apolipoprotein a1" and "lipoprotein apheresis".
LPA = (r"\blp\s*\(\s*a\s*\)"                          # lp(a), lp (a)
       r"|\blipoprotein\s*\(\s*a\s*\)"                # lipoprotein(a), lipoprotein (a)
       r"|\blipoprotein\s+a\b(?!\s*-?\s*(?:i|1)\b)"     # lipoprotein a  (not apo A-I style 'a-i' / 'a1')
       r"|\bapolipoprotein\s*\(\s*a\s*\)"             # apolipoprotein(a) = apo(a)
       r"|\blp-?a\b")                                  # lpa, lp-a
df["lpa"] = low.str.contains(LPA, regex=True)
lpa_posts = set("t3_" + df.loc[df.type_of_entry.eq("post") & df.lpa, "id"].astype(str))
df["lpa_thread"] = df["lpa"] | df["link_id"].isin(lpa_posts)      # sensitivity definition

# Promoted / recruitment content: Reddit ads metadata, sponsor domain, or campaign text
dom = df["domain_override"].fillna("").astype(str) if "domain_override" in df else pd.Series("", index=df.index)
ads = df["is_created_from_ads_ui"].fillna(0) if "is_created_from_ads_ui" in df else pd.Series(0, index=df.index)
camp = low.str.contains(r"careaccess|care access|know\s*my\s*lp|knowmylpa|national dataset about lp", regex=True)
df["promo"] = ads.eq(1) | dom.str.contains("careaccess|knowmylpa", case=False) | camp

# Most prolific 1% of accounts (for sensitivity analyses)
per_author = df.loc[df.author.fillna("[deleted]").ne("[deleted]"), "author"].value_counts()
top1 = set(per_author.head(max(1, int(0.01 * len(per_author)))).index)
df["top1_author"] = df.author.isin(top1)

primary_df = df[df.lpa & ~df.promo].copy()                        # topic-model input
# Recruitment-style posts that do not name the sponsor (retained; counted and reported)
RECRUIT = (r"gift card|free (?:cardiovascular|heart|cholesterol|lp\s*\(?a\)?|lipoprotein) (?:screening|test)"
           r"|research (?:company|study|site) (?:is )?(?:offering|recruiting|looking)|paid (?:study|research)"
           r"|compensat(?:ed|ion) for (?:your )?(?:time|participation)|participants? (?:needed|wanted)"
           r"|(?:we are|now) (?:recruiting|enrolling)|qualify for (?:a|our) (?:study|trial)")
_rec = primary_df["body"].fillna("").str.lower().str.contains(RECRUIT, regex=True)

y15 = df.year.eq(2015)
R["corpus"] = {
    "n_rows_retrieved": int(n_rows_retrieved), "n_duplicate_rows_removed": int(n_rows_retrieved - len(df)),
    "n_curated": int(len(df)),
    "n_lpa_explicit": int(df.lpa.sum()), "pct_lpa_explicit": round(100 * df.lpa.mean(), 1),
    "n_lpa_thread": int(df.lpa_thread.sum()), "pct_lpa_thread": round(100 * df.lpa_thread.mean(), 1),
    "n_promo": int(df.promo.sum()), "n_promo_lpa": int((df.promo & df.lpa).sum()),
    "promo_by_year": {int(k): int(v) for k, v in df[df.promo].groupby("year").size().items()},
    "n_primary_input": int(len(primary_df)),
    "n_recruitment_like_unflagged": int(_rec.sum()), "recruitment_like_by_year": {int(k): int(v) for k, v in primary_df.loc[_rec, "year"].value_counts().items()},
    "y2015": {"n": int(y15.sum()), "posts": int((y15 & df.type_of_entry.eq("post")).sum()),
              "comments": int((y15 & df.type_of_entry.eq("comment")).sum()),
              "pct_lpa": round(100 * df.loc[y15, "lpa"].mean(), 1),
              "n_lpa": int((y15 & df.lpa).sum()),
              "top_subreddits": df.loc[y15, "subreddit"].value_counts().head(6).to_dict()},
}
yrs = list(range(Y0, Y1 + 1))
full_y = df.groupby("year").size().reindex(yrs, fill_value=0)
lpa_y  = df[df.lpa].groupby("year").size().reindex(yrs, fill_value=0)
R["corpus"]["lpa_share_by_year_pct"] = {int(y): round(100 * lpa_y[y] / full_y[y], 1) for y in yrs if full_y[y]}

keep = ["id", "type_of_entry", "created_utc", "subreddit", "author", "link_id", "parent_id", "body",
        "promo", "top1_author", "lpa", "lpa_thread"]
df.loc[df.lpa, keep].to_json(f"{REV}/lpa_subset_strict.json", orient="records")
primary_df[keep].to_json(f"{REV}/lpa_subset_primary.json", orient="records")
print(json.dumps(R["corpus"], indent=1))

## 2. Keyword themes and precision sample (Table S5; R1.7)
Share of Lp(a)-explicit, non-promoted documents matching each keyword theme, with a random sample of matches per theme for manual precision scoring. The first four patterns are those used for the paraphrased excerpts in Table S6.

In [ ]:
THEMES = {
 "Cost / access / coverage": (r"not covered|out[- ]of[- ]pocket|pay(?:ing)? for|afford|copay|co-pay|deductible|\$\s?\d|charged|reimburs|\binsurance\b|\bmedicare\b|covered by", None, False),
 "Interpreting a result (uncertainty)": (r"is (?:this|it|that|my|an?) .{0,25}(?:bad|high|normal|concerning|serious|ok|dangerous)|what does .{0,20}mean|should i (?:be )?worr|can (?:someone|anyone) explain|do i need to worry|is (?:that|this) (?:good|bad|normal)", None, True),
 "Marker discordance (Lp(a) vs LDL/apoB)": (r"apob|apo b|\bldl\b", r"despite|even though|but my|but i|however|only my|normal|low|elevated", False),
 "Family / genetic risk & screening": (r"\bfamily\b|siblings?|\bkids\b|children|relatives?|parents?|father|mother|brother|sister|\bson\b|daughter|hereditary|inherited|genetic", r"test|screen|check", False),
 # supporting patterns
 "Any LDL-C or apoB mention": (r"apob|apo b|\bldl\b", None, False),
 "Established lipid-lowering therapy": (r"statin|pcsk9|repatha|praluent|evolocumab|alirocumab|niacin|ezetimibe|inclisiran", None, False),
 "Named investigational Lp(a) agent": (r"pelacarsen|olpasiran|lepodisiran|zerlasiran|muvalaplin", None, False),
}
base = df.lpa & ~df.promo
n = int(base.sum())
theme_rows, prec_rows = [], []
rng = np.random.default_rng(config.SEED)
for th, (pat, extra, need_q) in THEMES.items():
    m = base & low.str.contains(pat, regex=True)
    if extra is not None: m &= low.str.contains(extra, regex=True)
    if need_q: m &= low.str.contains(r"\?", regex=True)
    p = m.sum() / n; se = np.sqrt(p * (1 - p) / n)
    theme_rows.append({"theme": th, "n": int(m.sum()), "pct": round(100 * p, 1),
                       "ci95_lo": round(100 * (p - 1.96 * se), 1), "ci95_hi": round(100 * (p + 1.96 * se), 1)})
    idx = df.index[m]
    for i in rng.choice(idx, size=min(50, len(idx)), replace=False):
        f = re.sub(r"\s+", " ", txt.at[i]); mm = re.search(pat, f.lower())          # show the text around the matched keyword
        a, b = max(0, mm.start() - 450), min(len(f), mm.end() + 450)
        ctx = ("[...] " if a else "") + f[a:mm.start()] + "[[" + f[mm.start():mm.end()] + "]]" + f[mm.end():b] + (" [...]" if b < len(f) else "")
        ctx = re.sub(r"(?i)(?<![a-z0-9])/?u/[a-z0-9_-]{3,}", "u/[user]", ctx)
        prec_rows.append({"theme": th, "subreddit": show_sub(df.at[i, "subreddit"]), "text": mask_names(ctx), "true_positive (1/0)": ""})
themes_tbl = pd.DataFrame(theme_rows)
themes_tbl.to_csv(f"{REV}/theme_proportions.csv", index=False)
pd.DataFrame(prec_rows).to_csv(f"{REV}/theme_precision_sample.csv", index=False)
R["themes"] = {"denominator_lpa_docs": n, "rows": theme_rows}
print(f"denominator: {n} Lp(a)-explicit, non-promoted documents")
themes_tbl

In [ ]:
# Score the precision sample once 'true_positive (1/0)' is filled in and saved as theme_precision_sample_completed.csv
f = f"{REV}/theme_precision_sample_completed.csv"
if os.path.exists(f):
    pr = pd.read_csv(f).dropna(subset=["true_positive (1/0)"])
    def wilson(k, n, z=1.96):
        p = k / n; c = (p + z**2 / (2 * n)) / (1 + z**2 / n); h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
        return c - h, c + h
    raw = {r["theme"]: r["pct"] for r in R["themes"]["rows"]}
    rows_p = []
    for th, g in pr.groupby("theme")["true_positive (1/0)"]:
        k, m = int(g.sum()), int(g.size); lo, hi = wilson(k, m)
        rows_p.append({"theme": th, "n_reviewed": m, "precision": round(k / m, 2), "ci95": [round(lo, 2), round(hi, 2)],
                       "adjusted_pct": round(raw[th] * k / m, 1), "adjusted_range": [round(raw[th] * lo, 1), round(raw[th] * hi, 1)]})
    prec = pd.DataFrame(rows_p)                      # Table S5: Wilson 95% CI; prevalence = raw % x precision
    R["theme_precision"] = rows_p
    display(prec)
else:
    print("precision sample not scored yet:", f)

## 3. Nesting and repeat posters (S3, R2.3)
Account- and thread-level concentration of documents. Account names are not printed or saved.

In [ ]:
A = df[df.author.fillna("[deleted]").ne("[deleted]")]
L = df[df.lpa & ~df.promo]
uauth_y = L[L.author.fillna("[deleted]").ne("[deleted]")].groupby("year").author.nunique().reindex(yrs, fill_value=0)
R["nesting"] = {
    "distinct_accounts": int(A.author.nunique()), "deleted_author_docs": int(df.author.eq("[deleted]").sum()),
    "median_docs_per_account": float(per_author.median()), "max_docs_one_account": int(per_author.max()),
    "pct_docs_top1pct_accounts": round(100 * A.author.isin(top1).mean(), 1),
    "distinct_threads": int(df.link_id.nunique()),
    "lpa_unique_accounts_by_year": {int(y): int(v) for y, v in uauth_y.items()},
}
print(json.dumps({k: v for k, v in R["nesting"].items() if k != "lpa_unique_accounts_by_year"}, indent=1))
# Concentration within the analytic Lp(a) set (account names are never printed or saved)
la = L.loc[L.author.fillna("[deleted]").ne("[deleted]"), "author"].value_counts()
top1_lpa = la.head(max(1, int(0.01 * len(la))))
R["nesting_lpa"] = {"n_docs": int(len(L)), "distinct_accounts": int(la.size),
    "median_docs_per_account": float(la.median()),
    "top_account_docs": int(la.iloc[0]), "top_account_pct": round(100 * la.iloc[0] / len(L), 1),
    "top5_accounts_pct": round(100 * la.head(5).sum() / len(L), 1),
    "top1pct_accounts_n": int(len(top1_lpa)), "top1pct_accounts_pct": round(100 * top1_lpa.sum() / len(L), 1),
    "accounts_with_1_doc_pct": round(100 * float((la == 1).mean()), 1)}
print(json.dumps(R["nesting_lpa"], indent=1))
print("documents from the 10 most active accounts (anonymized):", la.head(10).tolist())

## 4. Descriptive growth (Table S9; S2)
Negative binomial trend in yearly counts (2015 excluded), with single-slope and piecewise (knot at 2022) fits. Section 12 gives the platform-adjusted estimate.

In [ ]:
import statsmodels.formula.api as smf

def nb_trend(series, offset=None, drop=(2015,), knot=2022):
    d = pd.DataFrame({"year": series.index, "y": series.values})
    if offset is not None: d["off"] = np.log(offset.reindex(d.year).values)
    d = d[~d.year.isin(drop)].copy()
    d["year_c"] = d.year - 2018; d["post"] = np.clip(d.year - knot, 0, None)
    pct = lambda v: round(100 * (np.exp(v) - 1), 1)
    out = {}
    for name, f, term in [("single", "y ~ year_c", "year_c"), ("pre_knot", "y ~ year_c + post", "year_c"),
                          ("added_post_knot", "y ~ year_c + post", "post")]:
        m = smf.negativebinomial(f, d, offset=d["off"] if offset is not None else None).fit(disp=0, maxiter=500)
        lo, hi = m.conf_int().loc[term]
        out[name] = {"pct_per_yr": pct(m.params[term]), "ci95": [pct(lo), pct(hi)]}
    return out

series = {
    "search corpus (docs)": full_y,
    "Lp(a)-explicit (docs)": lpa_y,
    "Lp(a) excl promoted": df[df.lpa & ~df.promo].groupby("year").size().reindex(yrs, fill_value=0),
    "Lp(a) excl top-1% accounts": df[df.lpa & ~df.top1_author].groupby("year").size().reindex(yrs, fill_value=0),
    "Lp(a) unique accounts": uauth_y,
}
R["growth_descriptive"] = {k: nb_trend(v) for k, v in series.items()}
R["growth_descriptive"]["Lp(a) share of search corpus (offset = log all documents)"] = nb_trend(lpa_y, offset=full_y)
for k, v in R["growth_descriptive"].items():
    print(f"{k:52s} single {v['single']['pct_per_yr']:6.1f}%/yr {v['single']['ci95']} | "
          f"post-2022 added {v['added_post_knot']['pct_per_yr']}%/yr")

## 5. Seeded primary topic model on the Lp(a)-explicit subset (R1.1, R1.3)

In [ ]:
import importlib.metadata as imd
vers = {}
for p in ["bertopic", "sentence-transformers", "umap-learn", "hdbscan", "scikit-learn", "torch",
          "llama_cpp_python", "transformers", "pandas", "numpy", "statsmodels"]:
    try: vers[p] = imd.version(p)
    except Exception: vers[p] = None
vers["python"] = sys.version.split()[0]
R["software_versions"] = vers
R["model_config"] = {k: getattr(config, k) for k in ["SEED", "EMBEDDING_MODEL", "LLM_REPO", "LLM_FILE", "LLM_TEMPERATURE",
                     "MIN_CLUSTER_SIZE", "MIN_SAMPLES", "N_NEIGHBORS", "MIN_DIST", "N_COMPONENTS", "UMAP_METRIC",
                     "NGRAM_RANGE", "MIN_TOPIC_SIZE", "OUTLIER_STRATEGY", "OUTLIER_THRESHOLD", "N_GROUPS"]}
print(json.dumps(vers, indent=1))

In [ ]:
# Embed once; embeddings are cached and reused by the fit and section 10.
if RESUME:
    print("RESUME: embeddings are on Drive; skipped")
else:
    _tmp = TopicModeler(enable_llm=False, seed=config.SEED)
    _texts = _tmp.preprocess_texts(primary_df)
    emb = np.load(EMB_P) if os.path.exists(EMB_P) else None
    if emb is None or len(emb) != len(_texts):
        emb = _tmp.embedding_model.encode(_texts, batch_size=64, show_progress_bar=True); np.save(EMB_P, emb)
    print(len(_texts), "documents embedded")

### Clustering settings (Table S2)
On the deduplicated corpus, the original settings (2-D UMAP, EOM cluster selection) placed 95% or more of documents in a single cluster at every min_cluster_size. Clustering settings are therefore selected by cross-seed stability using the rule stated in the next cell; the full grid is reported in Table S2.

In [ ]:
# Clustering diagnostic on the deduplicated corpus (no LLM; uses the cached embeddings).
# Prespecified selection rule: among settings giving 20-100 topics with no topic holding >25% of documents,
# choose the one with the highest mean cross-seed adjusted Rand index (ARI); ties go to the original settings (2-D UMAP, EOM).
CK_P = f"{REV}/R_checkpoint.json"
if RESUME:
    _saved = json.load(open(CK_P))
    for k, v in _saved.items():                                    # restore results from section 5 onward; sections 1-4 are recomputed
        if k == "model_config": R["model_config"] = {**v, **{kk: vv for kk, vv in R.get("model_config", {}).items() if kk not in v}}
        elif k not in R: R[k] = v
    _mc = R["model_config"]
    MCS_PRIMARY, NC_PRIMARY, SEL_PRIMARY = _mc["MIN_CLUSTER_SIZE"], _mc["N_COMPONENTS"], _mc["CLUSTER_SELECTION_METHOD"]
    print("RESUME: settings from checkpoint:", MCS_PRIMARY, NC_PRIMARY, SEL_PRIMARY)
else:
    SEEDS = [config.SEED, 1, 2]
    rows, labels = [], {}
    for nc in [2, 5]:
        red = {}
        for sd in SEEDS:
            t0 = time.time()
            red[sd] = umap.UMAP(n_neighbors=config.N_NEIGHBORS, n_components=nc, min_dist=config.MIN_DIST,
                                metric=config.UMAP_METRIC, random_state=sd).fit_transform(emb)
            print(f"UMAP {nc}-D seed {sd}: {time.time()-t0:.0f}s", flush=True)
        for sel, mcs in itertools.product(["eom", "leaf"], [15, 20, 25, 35, 50, 75]):
            lab = {sd: hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=config.MIN_SAMPLES, metric="euclidean",
                                       cluster_selection_method=sel).fit(red[sd]).labels_ for sd in SEEDS}
            l0 = lab[config.SEED]; ct = pd.Series(l0[l0 != -1]).value_counts()
            rows.append({"umap_dims": nc, "selection": sel, "min_cluster_size": mcs,
                         "topics": int(len(ct)), "outlier_pct": round(100 * float((l0 == -1).mean()), 1),
                         "largest_topic_pct": round(100 * float(ct.max() / len(l0)), 1) if len(ct) else None,
                         "ARI_mean": round(float(np.mean([adjusted_rand_score(lab[a], lab[b]) for a, b in itertools.combinations(SEEDS, 2)])), 3)})
            labels[(nc, sel, mcs)] = l0
            print(rows[-1], flush=True)
    diag = pd.DataFrame(rows); diag.to_csv(f"{OUT}/clustering_diagnostic_grid.csv", index=False)
    ok = diag[diag.topics.between(20, 100) & (diag.largest_topic_pct <= 25)].copy()
    ok["orig"] = (ok.umap_dims == 2) & (ok.selection == "eom")
    best = ok.sort_values(["ARI_mean", "orig"], ascending=False).head(1)
    print("\neligible settings:", len(ok)); print(ok.sort_values("ARI_mean", ascending=False).head(8).to_string(index=False))
    print("\nchosen:", best.to_dict("records"))
    l0 = labels[(2, "eom", 35)]; print("\n2-D EOM 35 cluster sizes:", pd.Series(l0).value_counts().to_dict())
    R["clustering_diagnostic"] = diag.to_dict("records")
    MCS_PRIMARY = int(best.min_cluster_size.iloc[0]); NC_PRIMARY = int(best.umap_dims.iloc[0]); SEL_PRIMARY = str(best.selection.iloc[0])
    R["model_config"].update({"MIN_CLUSTER_SIZE": MCS_PRIMARY, "N_COMPONENTS": NC_PRIMARY, "CLUSTER_SELECTION_METHOD": SEL_PRIMARY,
        "selection_rule": "among settings with 20-100 topics and no topic >25% of documents, highest mean cross-seed ARI (3 seeds)"})
    print("primary:", MCS_PRIMARY, NC_PRIMARY, SEL_PRIMARY)

In [ ]:
# Fit (or reload) the seeded primary topic model
if RESUME:
    modeler = TopicModeler(enable_llm=False, seed=config.SEED, min_cluster_size=MCS_PRIMARY, n_components=NC_PRIMARY)
    modeler.load_complete_state(META_P, MODEL_P)
    print("RESUME:", R.get("model"))
else:
    t0 = time.time()
    modeler = TopicModeler(enable_llm=RUN_LLM, seed=config.SEED, min_cluster_size=MCS_PRIMARY, n_components=NC_PRIMARY)
    modeler.hdbscan_model = hdbscan.HDBSCAN(min_cluster_size=MCS_PRIMARY, min_samples=config.MIN_SAMPLES,
                                            metric="euclidean", cluster_selection_method=SEL_PRIMARY)
    texts = modeler.preprocess_texts(primary_df)            # cleaning, >=25 chars, English-only
    emb = np.load(EMB_P); assert len(emb) == len(texts)    # same cached embeddings as the diagnostic and section 10
    modeler.topic_model = BERTopic(embedding_model=modeler.embedding_model, umap_model=modeler.umap_model,
                                   hdbscan_model=modeler.hdbscan_model, vectorizer_model=modeler.vectorizer_model,
                                   representation_model=modeler.representation_model, verbose=True)
    modeler.texts = texts
    modeler.topics, modeler.probs = modeler.topic_model.fit_transform(texts, emb)
    modeler.filter_small_topics(config.MIN_TOPIC_SIZE)
    R["model"] = {"n_after_preprocess": int(len(modeler.df)),
                  "n_topics_before_reduction": int(len(set(modeler.topics)) - (1 if -1 in modeler.topics else 0)),
                  "outlier_pct_before_reduction": round(100 * float(np.mean(np.array(modeler.topics) == -1)), 1)}
    modeler.save_complete_state(f"{REV}/metadata_primary_raw.pkl", f"{REV}/model_primary_raw.bertopic")
    modeler.reduce_outliers(strategy=config.OUTLIER_STRATEGY, threshold=config.OUTLIER_THRESHOLD, relabel=RUN_LLM)
    R["model"]["outlier_pct_after_reduction"] = round(100 * float(np.mean(np.array(modeler.topics) == -1)), 1)
    modeler.save_complete_state(META_P, MODEL_P)
    print(R["model"], f"| {time.time() - t0:.0f}s")

In [ ]:
# Save a checkpoint of R and list the topics
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o
def checkpoint():
    json.dump(_clean(R), open(CK_P, "w"), indent=1); print("checkpoint saved:", sorted(R))
checkpoint()
info = modeler.topic_model.get_topic_info()
cnt = pd.Series(modeler.topics).value_counts()
for t in sorted(cnt.index):
    if t == -1: continue
    print(t, cnt[t], "|", mask_names(info.loc[info.Topic == t, "Name"].values[0][:80]), "|",
          mask_names(", ".join(w for w, _ in modeler.topic_model.get_topic(t)[:8])))
print("outliers:", cnt.get(-1, 0))

### Number of thematic groups (Table S3)
k is chosen from 4-12 by the mean silhouette score of balanced k-means on topic c-TF-IDF vectors (outlier topic excluded) across five seeds; ties go to the smaller k.

In [ ]:
if RESUME:
    N_GROUPS_PRIMARY = R["model_config"].get("N_GROUPS") or len(json.load(open(GROUPS_P)).get("group_summaries", {}))
    R["model_config"]["N_GROUPS"] = N_GROUPS_PRIMARY; print("RESUME: k =", N_GROUPS_PRIMARY)
else:
    krows = []
    for k in range(4, 13):
        sils, labs = [], []
        for s in [42, 7, 123, 2024, 99]:
            tc = TopicClusterer(modeler, n_groups=k, seed=s, auto_group=False)
            _, e = tc._topic_embedding(); lab = tc._balanced_kmeans(e, k)
            sils.append(silhouette_score(e, lab)); labs.append(lab)
        krows.append({"k": k, "silhouette_mean": round(float(np.mean(sils)), 3),
                      "group_ARI_across_seeds": round(float(np.mean([adjusted_rand_score(a, b) for a, b in itertools.combinations(labs, 2)])), 3)})
    kt = pd.DataFrame(krows); kt.to_csv(f"{OUT}/k_selection_primary.csv", index=False)
    N_GROUPS_PRIMARY = int(kt.sort_values(["silhouette_mean", "k"], ascending=[False, True]).k.iloc[0])
    R["model_config"]["N_GROUPS"] = N_GROUPS_PRIMARY
    R["k_selection_primary"] = kt.to_dict("records")
    print(kt.to_string(index=False)); print("chosen k:", N_GROUPS_PRIMARY)

In [ ]:
if RESUME:
    labeler = GroupLabeler(modeler, preloaded_state=GROUPS_P)
else:
    labeler = GroupLabeler(modeler, n_groups=N_GROUPS_PRIMARY, seed=config.SEED, use_llm_labels=RUN_LLM)
    labeler.refresh_group_counts()
    labeler.save_group_summaries(GROUPS_P)
assert -1 not in labeler.topic_to_group, "outlier topic was grouped (rerun the grouping with the fix cell)"
for _g, _s in labeler.group_summaries.items():                 # strip stray quotes and periods from LLM group labels
    if _s.get("llm_group_label"):
        _s["llm_group_label"] = mask_names(re.sub(r"^[\s'\"]+|[\s'\".]+$", "", str(_s["llm_group_label"])))
cnt = pd.Series(modeler.topics).value_counts()
for g in sorted(set(labeler.topic_to_group.values())):
    ts = sorted(int(t) for t, gg in labeler.topic_to_group.items() if gg == g)
    print(g, int(sum(cnt.get(t, 0) for t in ts)), "|", ts, "|", mask_names(labeler.current_labels()[g]))
checkpoint()

## 6. Tables and figures

### Table 1: analyzed documents by year

In [ ]:
ad = modeler.df.copy()
ad["year"] = pd.to_datetime(pd.to_numeric(ad["created_utc"], errors="coerce"), unit="s", utc=True).dt.year
table1_years = ad.groupby(["year", "type_of_entry"]).size().unstack(fill_value=0)
for c in ("post", "comment"):
    if c not in table1_years.columns: table1_years[c] = 0
table1_years = table1_years[["post", "comment"]]
table1_years["total"] = table1_years.sum(axis=1)
table1_years.loc["TOTAL"] = table1_years.sum()
print(f'N = {len(ad)} | posts {int(ad.type_of_entry.eq("post").sum())} | '
      f'comments {int(ad.type_of_entry.eq("comment").sum())} | span {int(ad.year.min())}–{int(ad.year.max())}')
table1_years.to_csv(f"{OUT}/table1_by_year.csv")
table1_years

### Table 2: thematic groups

In [ ]:
counts = pd.Series(modeler.topics).value_counts()              # document count per topic after outlier reduction
info   = modeler.topic_model.get_topic_info()[["Topic", "Name"]]
total  = int(counts.sum())
t2g    = labeler.topic_to_group                                # {topic: 'Group_x'}
clean_lbl = lambda l: final_name(mask_names(re.sub(r"^[\s'\"]+|[\s'\".]+$", "", str(l))))   # strip stray quotes/periods from LLM labels
gname  = {g: clean_lbl(l) for g, l in labeler.current_labels().items()}

topic_label = {}
for s in labeler.group_summaries.values():
    if isinstance(s, dict) and s.get("topics") and s.get("topic_labels") \
       and len(s["topics"]) == len(s["topic_labels"]):
        topic_label.update({int(t): l for t, l in zip(s["topics"], s["topic_labels"])})
for _, r in info.iterrows():
    topic_label.setdefault(int(r["Topic"]), r["Name"])
topic_label = {t: clean_lbl(l) for t, l in topic_label.items()}

td = pd.DataFrame([{"topic": t, "n": int(n), "group": gname.get(t2g.get(t), t2g.get(t))}
                   for t, n in counts.items() if t != -1])
if td["group"].isna().any():
    print("WARNING: topics with no group mapping:", td[td["group"].isna()]["topic"].tolist())

table2_groups = (td.groupby("group")
            .apply(lambda d: pd.Series({
                "n": int(d["n"].sum()),
                "# topics": d["topic"].nunique(),
                "Representative topics": "; ".join(
                    TOPIC_FINAL.get(int(t), topic_label.get(t, str(t)))
                    for t in d.sort_values("n", ascending=False)["topic"].head(2))}))
            .reset_index().rename(columns={"group": "Group"}))
table2_groups["%"] = (100 * table2_groups["n"] / total).round(1)
table2_groups = table2_groups[["Group", "n", "%", "# topics", "Representative topics"]] \
            .sort_values("n", ascending=False).reset_index(drop=True)
n_out = int(counts.get(-1, 0))
print(f"Total {total} | categorized {total - n_out} | residual outliers {n_out} "
      f"({100*n_out/total:.1f}%) | groups {len(table2_groups)}")
table2_groups.to_csv(f"{OUT}/table2_groups.csv", index=False)
R["table2_groups"] = table2_groups[["Group", "n", "%", "# topics"]].to_dict("records")
table2_groups

### Table S4: topics, groups, labels, and coherence

In [ ]:
counts = pd.Series(modeler.topics).value_counts()
total  = int(counts.sum())
info   = modeler.topic_model.get_topic_info().set_index("Topic")
t2g    = labeler.topic_to_group
clean_lbl = lambda l: final_name(mask_names(re.sub(r"^[\s'\"]+|[\s'\".]+$", "", str(l))))   # strip stray quotes/periods from LLM labels
gname  = {g: clean_lbl(l) for g, l in labeler.current_labels().items()}
topic_label = {}
for s in labeler.group_summaries.values():
    if isinstance(s, dict) and s.get("topics") and s.get("topic_labels") \
       and len(s["topics"]) == len(s["topic_labels"]):
        topic_label.update({int(t): l for t, l in zip(s["topics"], s["topic_labels"])})
topic_llm = {t: mask_names(re.sub(r"^[\s'\"]+|[\s'\".]+$", "", str(l))) for t, l in topic_label.items()}   # LLM-generated labels
topic_label = {t: clean_lbl(l) for t, l in topic_label.items()}

def keywords(t, k=10):
    try:    return mask_names(", ".join(w for w, _ in modeler.topic_model.get_topic(t)[:k] if w))
    except Exception: return ""

def exemplar(t, n=240):
    try:
        docs = modeler.topic_model.get_representative_docs(t)
        d = re.sub(r"(?i)(?<![a-z0-9])/?u/[a-z0-9_-]{3,}", "u/[user]", docs[0]) if docs else ""   # mask Reddit usernames
        d = mask_names(d)
        return (d[:n] + "…") if d else ""
    except Exception: return ""

_auth = modeler.df["author"].fillna("[deleted]").values; _tp = np.array(modeler.topics)
def top_account_pct(t):                              # share of a topic's documents written by its single most active account
    a = pd.Series(_auth[_tp == t]); a = a[~a.isin(["[deleted]", "[removed]"])]
    return round(100 * a.value_counts().iloc[0] / len(a), 1) if len(a) else None

rows = []
for t in sorted(x for x in counts.index if x != -1):
    g = t2g.get(t)
    nm = info.loc[t, "Name"] if t in info.index else str(t)
    rows.append({"Topic": int(t), "LLM-generated topic label": topic_llm.get(int(t), nm),
                 "Final topic label": TOPIC_FINAL.get(int(t), topic_label.get(int(t), nm)), "Parent group": gname.get(g, g),
                 "n": int(counts.get(t, 0)), "%": round(100 * int(counts.get(t, 0)) / total, 2),
                 "Top keywords": keywords(t), "Top account %": top_account_pct(t), "Exemplar discussion": exemplar(t)})
tableS4 = (pd.DataFrame(rows).sort_values(["Parent group", "n"], ascending=[True, False]).reset_index(drop=True))
n_out = int(counts.get(-1, 0))
print(f"{len(tableS4)} topics | total {total} | residual outliers {n_out} ({100*n_out/total:.1f}%)")
tableS4.to_csv(f"{OUT}/tableS4_topics.csv", index=False)
R["model"]["n_topics_final"] = int(len(tableS4))
tableS4

### Figure 1A and 1B

In [ ]:
fig1a_temporal(modeler.df, out_path=f'{OUT}/fig1a_temporal_lpa.png')
fig1b, emb2d, labels = fig1b_clusters(modeler, labeler, out_path=f'{OUT}/fig1b_clusters_lpa.png')

import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

def recolor_group(fig, old_label, new_color, new_label=None, tol=0.02):
    # recolor one group's points and legend entry in Fig 1B (keeps alpha); optionally rename it
    ax = fig.axes[0]
    leg = ax.get_legend() or (fig.legends[0] if fig.legends else None)
    handles = getattr(leg, "legend_handles", None) or leg.legendHandles
    for t, h in zip(leg.get_texts(), handles):
        if t.get_text() != old_label:
            continue
        old = np.array(mcolors.to_rgb(h.get_markerfacecolor() if isinstance(h, Line2D) else h.get_facecolor()[0]))
        new = np.array(mcolors.to_rgb(new_color))
        for c in ax.collections:
            fc = c.get_facecolors()
            if len(fc) == 0: continue
            hit = np.all(np.abs(fc[:, :3] - old) < tol, axis=1)
            if hit.any():
                fc[hit, :3] = new; c.set_facecolors(fc)
                ec = c.get_edgecolors()
                if len(ec) == len(fc): ec[hit, :3] = new; c.set_edgecolors(ec)
        if isinstance(h, Line2D): h.set_markerfacecolor(new_color); h.set_markeredgecolor(new_color)
        else: h.set_facecolor(new_color); h.set_edgecolor(new_color)
        if new_label: t.set_text(new_label)
        return True
    print("label not found:", old_label); return False

if hasattr(fig1b, 'findobj'):                                   # apply final names and colors, then re-save
    relabel_figure(fig1b)
    recolor_group(fig1b, "Other", "#D9D9D9", "Unassigned")
    recolor_group(fig1b, "Non-statin and combination lipid-lowering therapy", "#8C564B")
    for ext in ("png", "pdf", "tiff"):
        fig1b.savefig(f'{OUT}/fig1b_clusters_lpa.{ext}', dpi=300, bbox_inches='tight')
fig1b


### Figure S3: search corpus and Lp(a)-explicit documents by year (R1.5)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
x = np.array(yrs)
ax.bar(x - 0.2, full_y.values, width=0.4, color="#BDBDBD", label="All retrieved documents")
ax.bar(x + 0.2, lpa_y.values, width=0.4, color="#08519C", label="Explicit Lp(a) mention")
ax.set_ylabel("Documents per year"); ax.set_xticks(x); ax.tick_params(axis="x", rotation=45, labelsize=8)
ax2 = ax.twinx()
ax2.plot(x, 100 * lpa_y.values / np.maximum(full_y.values, 1), color="#D62728", marker="o", ms=3, label="Lp(a) share (%)")
ax2.set_ylabel("Lp(a) share of retrieved (%)"); ax2.set_ylim(0, 100)
for sp in ("top",): ax.spines[sp].set_visible(False); ax2.spines[sp].set_visible(False)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8, frameon=False)
fig.tight_layout()
for ext in ("png", "pdf"): fig.savefig(f"{OUT}/SuppFig_full_vs_lpa.{ext}", dpi=300, bbox_inches="tight")

### Figure S2: subreddit distribution of analyzed documents

In [ ]:
d = modeler.df.dropna(subset=["subreddit"]).copy(); d["subreddit"] = d["subreddit"].map(show_sub); N = len(d); U = d["subreddit"].nunique()
tot = d["subreddit"].value_counts(); top = tot.head(20).index[::-1]
pc = (d[d.subreddit.isin(top)].groupby(["subreddit","type_of_entry"]).size()
        .unstack(fill_value=0).reindex(top))
for c in ("post","comment"):
    if c not in pc: pc[c] = 0
fig, ax = plt.subplots(figsize=(8.2, 7.4)); y = np.arange(len(top))
ax.barh(y, pc["comment"], color="#6BAED6", label="comment", edgecolor="white", linewidth=0.4)
ax.barh(y, pc["post"], left=pc["comment"], color="#08519C", label="post", edgecolor="white", linewidth=0.4)
for i, s in enumerate(top):
    n_s = int(tot[s]); ax.text(n_s + N*0.004, i, f"{n_s:,} ({100*n_s/N:.1f}%)", va="center", fontsize=8)
ax.set_yticks(y); ax.set_yticklabels([s if s in SUBREDDIT_MASK.values() else "r/"+s for s in top], fontsize=9)
ax.set_xlabel("Number of discussions (posts + comments)")
ax.set_title("Distribution of Lp(a) discussions across subreddits (top 20)", loc="left", fontweight="bold")
ax.legend(title="Content type", loc="lower right"); ax.set_xlim(0, int(tot[top].max())*1.18)
for sp in ("top","right"): ax.spines[sp].set_visible(False)
ax.grid(axis="x", alpha=0.25)
ax.text(0.99, -0.085, f"{U:,} unique subreddits; top 20 account for {100*tot.head(20).sum()/N:.1f}% of {N:,} discussions.",
        transform=ax.transAxes, ha="right", va="top", fontsize=7.5, color="#555555")
fig.tight_layout(); fig.savefig(f"{OUT}/SuppFig_subreddits.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{OUT}/SuppFig_subreddits.pdf", bbox_inches="tight")
R["subreddits"] = {"n_unique": int(U), "top20_pct": round(100*tot.head(20).sum()/N, 1),
                   "top10": {k: int(v) for k, v in tot.head(10).items()}}

## 7. Thematic group share over time (Figure 2, Table S7; R1.10)
Share of each thematic group among topic-assigned documents per year (2019-2024, years with at least 100 documents); milestones within the window are marked (the 2022 EAS consensus statement). A quasi-binomial trend per group (Pearson-scaled SE) gives an odds ratio per year for the group's share.

In [ ]:
clean_lbl = lambda l: final_name(mask_names(re.sub(r"^[\s'\"]+|[\s'\".]+$", "", str(l))))
glabels, _ = labeler.create_labels_for_visualization()          # aligned with modeler.topics / modeler.df
gt = pd.DataFrame({"year": pd.to_datetime(pd.to_numeric(modeler.df["created_utc"]), unit="s", utc=True).dt.year.values,
                   "group": [clean_lbl(g) for g in glabels], "topic": modeler.topics})
gt = gt[(gt.topic != -1) & gt.year.between(Y0, Y1)]
MIN_N = 100                                                        # use only years with enough categorized documents
share = gt.groupby(["year", "group"]).size().unstack(fill_value=0)
share = share[share.sum(axis=1) >= MIN_N]
prop = share.div(share.sum(axis=1), axis=0)
prop.round(4).to_csv(f"{OUT}/group_share_by_year.csv")
print("years used:", list(share.index), "| categorized docs per year:", share.sum(axis=1).tolist())

_TOL = ["#4477AA","#EE6677","#228833","#CCBB44","#66CCEE","#AA3377","#BBBBBB","#EE8866","#77AADD","#999933"]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.stackplot(prop.index, (100 * prop.T).values, labels=prop.columns, colors=_TOL[:len(prop.columns)], alpha=0.9)
for yr, lab, kind in config.MILESTONES:
    if prop.index.min() < yr < prop.index.max():                  # interior milestones only
        ax.axvline(yr, color="k", ls="--", lw=0.8); ax.text(yr, 101, str(yr), ha="center", fontsize=7)
ax.set_xlim(prop.index.min(), prop.index.max()); ax.set_ylim(0, 100)
ax.set_ylabel("Share of topic-assigned documents (%)"); ax.set_xlabel("Year")
h, l = ax.get_legend_handles_labels()                             # legend order matches stack order
ax.legend(h[::-1], l[::-1], loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=7, frameon=False)
fig.tight_layout()
for ext in ("png", "pdf", "tiff"): fig.savefig(f"{OUT}/Fig_group_share_over_time.{ext}", dpi=300, bbox_inches="tight")

def quasi_binomial(years, k, n, center=2018):
    # grouped logistic regression by IRLS; SE scaled by Pearson dispersion (floored at 1)
    x = np.asarray(years, float) - center; X = np.column_stack([np.ones_like(x), x])
    k = np.asarray(k, float); n = np.asarray(n, float)
    beta = np.array([np.log((k.sum() + .5) / (n.sum() - k.sum() + .5)), 0.0])
    for _ in range(200):
        p = 1 / (1 + np.exp(-(X @ beta))); w = n * p * (1 - p)
        z = X @ beta + (k - n * p) / np.maximum(w, 1e-12)
        new = np.linalg.solve(X.T @ (w[:, None] * X), X.T @ (w * z))
        if np.max(np.abs(new - beta)) < 1e-10: beta = new; break
        beta = new
    p = 1 / (1 + np.exp(-(X @ beta))); w = n * p * (1 - p)
    phi = max(1.0, float(np.sum((k - n * p) ** 2 / np.maximum(w, 1e-12)) / (len(k) - 2)))
    se = float(np.sqrt(phi * np.linalg.inv(X.T @ (w[:, None] * X))[1, 1]))
    return float(np.exp(beta[1])), [float(np.exp(beta[1] - 1.96 * se)), float(np.exp(beta[1] + 1.96 * se))], phi

trend = []
N = share.sum(axis=1)
for g in share.columns:
    orr, ci, phi = quasi_binomial(share.index, share[g], N)
    trend.append({"group": g, "n_docs": int(share[g].sum()),
                  f"share_{share.index[0]}": round(100 * prop[g].iloc[0], 1), f"share_{share.index[-1]}": round(100 * prop[g].iloc[-1], 1),
                  "OR_per_yr": round(orr, 3), "ci95": [round(ci[0], 3), round(ci[1], 3)], "dispersion": round(phi, 1)})
R["group_share_trend"] = {"years": [int(share.index[0]), int(share.index[-1])], "model": "quasi-binomial, Pearson-scaled SE", "rows": trend}
pd.set_option("display.max_colwidth", 60)
pd.DataFrame(trend).sort_values("OR_per_yr", ascending=False)

### Sensitivity: excluding the most active 1% of accounts, and alternative windows

In [ ]:
# Group-share trends excluding documents from the most active 1% of accounts (same years and MIN_N as above)
t1 = set(top1)
keep = ~modeler.df["author"].isin(t1).values
gt2 = pd.DataFrame({"year": pd.to_datetime(pd.to_numeric(modeler.df["created_utc"]), unit="s", utc=True).dt.year.values,
                    "group": [clean_lbl(g) for g in glabels], "topic": modeler.topics})[keep]
gt2 = gt2[(gt2.topic != -1) & gt2.year.between(Y0, Y1)]
sh2 = gt2.groupby(["year", "group"]).size().unstack(fill_value=0); sh2 = sh2[sh2.sum(axis=1) >= MIN_N]
N2 = sh2.sum(axis=1); rows2 = []
for g in sh2.columns:
    orr, ci, phi = quasi_binomial(sh2.index, sh2[g], N2)
    rows2.append({"group": g[:60], "n_docs": int(sh2[g].sum()), "OR_per_yr": round(orr, 3), "ci95": [round(ci[0], 3), round(ci[1], 3)]})
R["group_share_trend_excl_top1pct"] = rows2
print("years:", list(sh2.index)); print(pd.DataFrame(rows2).sort_values("OR_per_yr", ascending=False).to_string(index=False))

In [ ]:
# Trend table (Table S7): all accounts vs excluding the top 1%, for 2019-2024 and 2021-2024 (years with >= 100 documents)
def trend_tbl(mask, y0, y1, min_n):
    g = pd.DataFrame({"year": pd.to_datetime(pd.to_numeric(modeler.df["created_utc"]), unit="s", utc=True).dt.year.values,
                      "group": [clean_lbl(x)[:45] for x in glabels], "topic": modeler.topics})[mask]
    g = g[(g.topic != -1) & g.year.between(y0, y1)]
    sh = g.groupby(["year", "group"]).size().unstack(fill_value=0); sh = sh[sh.sum(axis=1) >= min_n]
    out = {}
    for c in sh.columns:
        orr, ci, _ = quasi_binomial(sh.index, sh[c], sh.sum(axis=1)); out[c] = f"{orr:.2f} ({ci[0]:.2f}-{ci[1]:.2f})"
    return pd.Series(out), list(sh.index)
allm = np.ones(len(modeler.df), bool); notop = ~modeler.df["author"].isin(set(top1)).values
cols = {}
for lab, m in [("all", allm), ("excl_top1pct", notop)]:
    for (y0, y1, mn) in [(2019, 2024, 100), (2021, 2024, 100)]:
        s, yrs_used = trend_tbl(m, y0, y1, mn); cols[f"{lab} {yrs_used[0]}-{yrs_used[-1]}"] = s
sens_tbl = pd.DataFrame(cols); R["group_trend_sensitivity"] = sens_tbl.to_dict(); print(sens_tbl.to_string())

## 8. Topic coherence: NPMI (Table S4; R1.4)

In [ ]:
docs_l = [str(t).lower() for t in modeler.texts]
tok_sets = [set(re.findall(r"[a-z0-9]+", d)) for d in docs_l]
Nd = len(docs_l)
def occ(w):
    return {i for i, s in enumerate(tok_sets) if w in s} if " " not in w else {i for i, d in enumerate(docs_l) if w in d}
coh = []
for t in sorted(set(modeler.topics) - {-1}):
    words = [w for w, _ in modeler.topic_model.get_topic(t)[:10] if w]
    O = {w: occ(w) for w in words}
    sc = []
    for a, b in itertools.combinations(words, 2):
        pab = len(O[a] & O[b]) / Nd
        if pab == 0: sc.append(-1.0); continue
        pa, pb = len(O[a]) / Nd, len(O[b]) / Nd
        sc.append(1.0 if pab == 1 else np.log(pab / (pa * pb)) / -np.log(pab))
    coh.append({"topic": int(t), "npmi": round(float(np.mean(sc)), 3)})
coh = pd.DataFrame(coh); coh.to_csv(f"{OUT}/topic_npmi.csv", index=False)
R["coherence"] = {"mean_npmi": round(float(coh.npmi.mean()), 3), "median_npmi": round(float(coh.npmi.median()), 3),
                  "n_topics_npmi_below_0": int((coh.npmi < 0).sum())}
R["coherence"]["npmi_iqr"] = [round(float(coh.npmi.quantile(.25)), 3), round(float(coh.npmi.quantile(.75)), 3)]
print(R["coherence"])

# Single-word NPMI (standard form; avoids penalizing overlapping multi-word terms)
def npmi_unigram(t, k=10):
    words = []
    for w, _ in modeler.topic_model.get_topic(t):
        for u in w.split():
            if u not in words: words.append(u)
        if len(words) >= k: break
    words = words[:k]
    O = {w: {i for i, s in enumerate(tok_sets) if w in s} for w in words}
    sc = []
    for a, b in itertools.combinations(words, 2):
        pab = len(O[a] & O[b]) / Nd
        if pab == 0: sc.append(-1.0); continue
        pa, pb = len(O[a]) / Nd, len(O[b]) / Nd
        sc.append(1.0 if pab == 1 else np.log(pab / (pa * pb)) / -np.log(pab))
    return float(np.mean(sc))
cu = pd.DataFrame([{"topic": int(t), "npmi_unigram": round(npmi_unigram(t), 3)} for t in sorted(set(modeler.topics) - {-1})])
cu.to_csv(f"{OUT}/topic_npmi_unigram.csv", index=False)
R["coherence"]["unigram"] = {"median": round(float(cu.npmi_unigram.median()), 3),
                             "iqr": [round(float(cu.npmi_unigram.quantile(.25)), 3), round(float(cu.npmi_unigram.quantile(.75)), 3)],
                             "n_below_0": int((cu.npmi_unigram < 0).sum())}
print(R["coherence"]["unigram"])

In [ ]:
# Save a checkpoint of R
checkpoint()

## 9. Clinician label validation (Table S11; R1.2, R1.4)
Writes one blinded Excel workbook per rater (six random documents per topic; identical items and order) and a separate answer key. Completed workbooks saved as `rater1_completed.xlsx` and `rater2_completed.xlsx` in the output folder are scored in the next cell (agreement and Cohen's kappa).

In [ ]:
from openpyxl import Workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Alignment, Font

N_PER_TOPIC = 6
def rater_text(t, cap=1500):
    t = re.sub(r"&#x200[bB];|&nbsp;", " ", str(t))
    t = re.sub(r"(?i)(?<![a-z0-9])/?u/[a-z0-9_-]{3,}", "u/[user]", t)          # mask Reddit usernames
    t = mask_names(re.sub(r"\s+", " ", t).strip())
    if len(t) <= cap: return t
    m = re.search(LPA, t.lower())
    if m is None or m.start() < cap - 150: return t[:cap] + " [...]"
    return t[:700] + " [...] " + t[max(700, m.start() - 350): m.start() + 450] + " [...]"   # keep the Lp(a) mention visible
# Raters judged the LLM-generated topic and group labels (not the adjudicated names)
_strip = lambda l: mask_names(re.sub(r"^[\s'\"]+|[\s'\".]+$", "", str(l)))
gname_llm = {g: _strip(l) for g, l in labeler.current_labels().items()}
rng = np.random.default_rng(config.SEED)
tp = np.array(modeler.topics)
items, key = [], []
for t in sorted(set(modeler.topics) - {-1}):
    idx = np.where(tp == t)[0]
    for i in rng.choice(idx, size=min(N_PER_TOPIC, len(idx)), replace=False):
        items.append({"topic_id": int(t), "topic_label": topic_llm.get(int(t), str(t)),
                      "document": rater_text(modeler.texts[i])})
        key.append({"topic_id": int(t), "doc_index": int(i)})
order = rng.permutation(len(items))
items = [items[j] for j in order]; key = [key[j] for j in order]
for n, it in enumerate(items, 1): it["item_id"] = n
pd.DataFrame([{**{"item_id": n}, **k} for n, k in enumerate(key, 1)]).to_csv(f"{REV}/rater_answer_key.csv", index=False)

topics_tab = [{"topic_id": int(t), "topic_label": topic_llm.get(int(t), str(t)), "keywords": keywords(int(t)),
               "group_label": gname_llm.get(t2g.get(int(t)), "")} for t in sorted(set(modeler.topics) - {-1})]
groups_tab = []
for g, s in labeler.group_summaries.items():
    groups_tab.append({"group_id": g, "group_label": gname_llm.get(g, g),
                       "member_topics": "; ".join(topic_llm.get(int(t), str(t)) for t in s.get("topics", []))})

def write_rater(path):
    wb = Workbook()
    ws = wb.active; ws.title = "Documents"
    head = ["item_id", "topic_id", "topic_label", "document", "Q1_fits_topic_label", "Q2_about_lpa",
            "Q3_spam_or_offtopic", "comment"]
    ws.append(head)
    for it in items:
        ws.append([it["item_id"], it["topic_id"], it["topic_label"], it["document"], "", "", "", ""])
    yn = DataValidation(type="list", formula1='"Yes,No"', allow_blank=True)
    ws.add_data_validation(yn); yn.add(f"E2:G{len(items)+1}")
    for col, w in zip("ABCDEFGH", [8, 8, 30, 90, 14, 12, 14, 30]): ws.column_dimensions[col].width = w
    for row in ws.iter_rows(min_row=2):
        for c in row: c.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "E2"
    wt = wb.create_sheet("Topics")
    wt.append(["topic_id", "topic_label", "keywords", "Q4_label_accurate", "suggested_label", "comment"])
    for t in topics_tab: wt.append([t["topic_id"], t["topic_label"], t["keywords"], "", "", ""])
    yn2 = DataValidation(type="list", formula1='"Yes,No"', allow_blank=True); wt.add_data_validation(yn2); yn2.add(f"D2:D{len(topics_tab)+1}")
    for col, w in zip("ABCDEF", [8, 40, 50, 14, 35, 30]): wt.column_dimensions[col].width = w
    wg = wb.create_sheet("Groups")
    wg.append(["group_id", "group_label", "member_topics", "Q5_label_accurate", "suggested_label", "comment"])
    for g in groups_tab: wg.append([g["group_id"], g["group_label"], g["member_topics"], "", "", ""])
    yn3 = DataValidation(type="list", formula1='"Yes,No"', allow_blank=True); wg.add_data_validation(yn3); yn3.add(f"D2:D{len(groups_tab)+1}")
    for col, w in zip("ABCDEF", [10, 40, 80, 14, 35, 30]): wg.column_dimensions[col].width = w
    for sh in (ws, wt, wg):
        for c in sh[1]: c.font = Font(bold=True)
    wb.save(path)

for r in (1, 2): write_rater(f"{REV}/rater{r}_sheet.xlsx")
print(f"{len(items)} documents, {len(topics_tab)} topics, {len(groups_tab)} groups -> rater1_sheet.xlsx, rater2_sheet.xlsx")
print("rater_answer_key.csv is for the adjudicator only; do not share with raters")

In [ ]:
f1, f2 = f"{REV}/rater1_completed.xlsx", f"{REV}/rater2_completed.xlsx"
if os.path.exists(f1) and os.path.exists(f2):
    from sklearn.metrics import cohen_kappa_score
    yn = {"yes": 1, "no": 0}
    def load(f, sheet):
        d = pd.read_excel(f, sheet_name=sheet)
        for c in d.columns:
            if c.startswith("Q"): d[c] = d[c].astype(str).str.strip().str.lower().map(yn)
        return d
    D1, D2 = load(f1, "Documents"), load(f2, "Documents")
    D = D1.merge(D2, on=["item_id", "topic_id", "topic_label"], suffixes=("_r1", "_r2"))
    out = {"n_docs_rated": int(len(D))}
    for q in ["Q1_fits_topic_label", "Q2_about_lpa", "Q3_spam_or_offtopic"]:
        d = D.dropna(subset=[f"{q}_r1", f"{q}_r2"])
        a, b = d[f"{q}_r1"].astype(int), d[f"{q}_r2"].astype(int)
        out[q] = {"n": int(len(d)), "rater1_yes": round(a.mean(), 3), "rater2_yes": round(b.mean(), 3),
                  "both_yes": round(((a + b) == 2).mean(), 3), "pct_agree": round((a == b).mean(), 3),
                  "kappa": round(cohen_kappa_score(a, b), 2) if a.nunique() > 1 or b.nunique() > 1 else None}
    D["consensus_fit"] = ((D.Q1_fits_topic_label_r1 + D.Q1_fits_topic_label_r2) == 2).astype(int)
    per = D.groupby(["topic_id", "topic_label"]).consensus_fit.mean().reset_index(name="fit_rate")
    out["topics_fit_below_0.6"] = per[per.fit_rate < 0.6].topic_id.tolist()
    T = load(f1, "Topics").merge(load(f2, "Topics"), on=["topic_id", "topic_label"], suffixes=("_r1", "_r2"))
    G = load(f1, "Groups").merge(load(f2, "Groups"), on=["group_id", "group_label"], suffixes=("_r1", "_r2"))
    for q, tab in [("Q4_label_accurate", T), ("Q5_label_accurate", G)]:
        out[q] = {"rater1_yes": int(tab[f"{q}_r1"].sum()), "rater2_yes": int(tab[f"{q}_r2"].sum()), "n": int(len(tab))}
    out["topic_labels_both_accurate"] = round(((T.Q4_label_accurate_r1 + T.Q4_label_accurate_r2) == 2).mean(), 3)
    out["group_labels_both_accurate"] = round(((G.Q5_label_accurate_r1 + G.Q5_label_accurate_r2) == 2).mean(), 3)
    R["label_validation"] = out
    dis = D[(D.Q1_fits_topic_label_r1 != D.Q1_fits_topic_label_r2) | (D.Q3_spam_or_offtopic_r1 != D.Q3_spam_or_offtopic_r2)]
    dis.to_csv(f"{REV}/rater_disagreements.csv", index=False)
    T.to_csv(f"{REV}/rater_topic_labels.csv", index=False); G.to_csv(f"{REV}/rater_group_labels.csv", index=False)
    print(json.dumps(out, indent=1)); print(len(dis), "disagreements -> rater_disagreements.csv (adjudicate)")
    display(per[per.fit_rate < 0.6])
else:
    print("waiting for rater1_completed.xlsx and rater2_completed.xlsx")

## 10. Sensitivity, stability, and choice of k (Table S10; R1.3, S3)
Refits UMAP and HDBSCAN on the cached embeddings across a parameter grid and five seeds, reporting topic counts, outlier share, and cross-seed adjusted Rand index (ARI). The next cell plots the silhouette score across k = 2-20 (exploratory; k was selected over 4-12 in section 5).

In [ ]:
if RUN_SENSITIVITY:
    texts = list(modeler.texts)
    if os.path.exists(EMB_P):
        emb = np.load(EMB_P)
    else:
        emb = modeler.embedding_model.encode(texts, batch_size=64, show_progress_bar=True); np.save(EMB_P, emb)
    SEEDS = [42, 7, 123, 2024, 99]; NN = [5, 10, 15, 30]; MCS = sorted({MCS_PRIMARY // 2, MCS_PRIMARY, 2 * MCS_PRIMARY, 100})
    def fit_once(nn, mcs, seed):
        m = BERTopic(umap_model=umap.UMAP(n_neighbors=nn, n_components=NC_PRIMARY, min_dist=config.MIN_DIST,
                                          metric=config.UMAP_METRIC, random_state=seed),
                     hdbscan_model=hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=config.MIN_SAMPLES,
                                                   metric="euclidean", cluster_selection_method=SEL_PRIMARY),
                     vectorizer_model=CountVectorizer(stop_words="english", ngram_range=config.NGRAM_RANGE), verbose=False)
        t, _ = m.fit_transform(texts, emb); return m, np.array(t)
    sens, primary_runs = [], {}
    for nn, mcs in itertools.product(NN, MCS):
        runs = []
        for s in SEEDS:
            m, t = fit_once(nn, mcs, s); runs.append(t)
            if nn == config.N_NEIGHBORS and mcs == MCS_PRIMARY: primary_runs[s] = m
        aris = [adjusted_rand_score(a, b) for a, b in itertools.combinations(runs, 2)]
        sens.append({"n_neighbors": nn, "min_cluster_size": mcs,
                     "topics_mean": round(float(np.mean([len(set(r)) - (-1 in r) for r in runs])), 1),
                     "topics_min": int(min(len(set(r)) - (-1 in r) for r in runs)),
                     "topics_max": int(max(len(set(r)) - (-1 in r) for r in runs)),
                     "outlier_pct": round(100 * float(np.mean([(r == -1).mean() for r in runs])), 1),
                     "ARI_mean": round(float(np.mean(aris)), 3), "ARI_min": round(float(np.min(aris)), 3)})
        print(sens[-1], flush=True)
    sens = pd.DataFrame(sens); sens.to_csv(f"{OUT}/sensitivity_grid.csv", index=False)
    R["sensitivity"] = sens.to_dict("records")
    display(sens)

In [ ]:
if RUN_SENSITIVITY:
    class _Shim:
        def __init__(self, m): self.topic_model = m
    krows = []
    for k in range(4, 13):
        sils, labs = [], []
        for s in SEEDS:
            tc = TopicClusterer(modeler, n_groups=k, seed=s, auto_group=False)      # seeded primary topics
            _, e = tc._topic_embedding(); lab = tc._balanced_kmeans(e, k)
            sils.append(silhouette_score(e, lab)); labs.append(lab)
        krows.append({"k": k, "silhouette_mean": round(float(np.mean(sils)), 3),
                      "group_ARI_across_seeds": round(float(np.mean([adjusted_rand_score(a, b)
                                                                      for a, b in itertools.combinations(labs, 2)])), 3)})
        print(krows[-1], flush=True)
    kt = pd.DataFrame(krows); kt.to_csv(f"{OUT}/k_selection.csv", index=False)
    R["k_selection"] = kt.to_dict("records")
    if fig_silhouette_scan is not None:
        fig_silhouette_scan(TopicClusterer(modeler, auto_group=False), k_min=2, k_max=20, out_path=f'{OUT}/fig_silhouette_scan.png')
    display(kt)

## 11. Platform denominators: Arctic Shift (R1.5, S1, R2.2)
Total yearly posts and comments (all topics) in the ten subreddits with the most Lp(a) documents, from the Arctic Shift archive. Unexpected API responses are printed.

In [ ]:
import requests
SUBS = primary_df.subreddit.value_counts().head(10).index.tolist()
R["denominator_subreddits"] = SUBS
R["denominator_coverage_pct"] = round(100 * primary_df.subreddit.isin(SUBS).mean(), 1)
DEN_P = f"{REV}/denominators.csv"
def yearly_totals(sub, kind):
    # Precomputed yearly counts (the search/aggregate endpoint times out on busy subreddits)
    r = requests.get("https://arctic-shift.photon-reddit.com/api/time_series",
                     params={"key": f"r/{sub}/{kind}/count", "precision": "year"}, timeout=120)
    r.raise_for_status(); js = r.json(); data = js.get("data") if isinstance(js, dict) else None
    if data is None: print(f"[{sub}/{kind}] unexpected response:", str(js)[:300]); return {}
    out = {}
    for row in data:
        y = pd.to_datetime(int(row["date"]), unit="s", utc=True).year
        out[y] = out.get(y, 0) + int(row.get("value") or 0)
    return out
if RUN_DENOMINATORS:
    rows = []
    for sub in SUBS:
        for kind in ("posts", "comments"):
            try: tot_ = yearly_totals(sub, kind)
            except Exception as e: print(f"[{sub}/{kind}] failed: {e}"); tot_ = {}
            rows += [{"subreddit": sub, "year": y, "kind": kind, "total": tot_.get(y, 0 if tot_ else None)} for y in range(Y0, 2026)]
            print(show_sub(sub), kind, {y: tot_.get(y) for y in (2011, 2018, 2024)}, flush=True); time.sleep(1.0)
    den = pd.DataFrame(rows); den.to_csv(DEN_P, index=False)
print("covers", R["denominator_coverage_pct"], "% of primary Lp(a) documents:", [show_sub(x) for x in SUBS])

## 12. Platform-adjusted negative binomial model (Table S9; R1.5, S1, S2, R2.2)
Yearly Lp(a) documents per subreddit with log(total subreddit activity) as offset, subreddit fixed effects, and 2015 excluded. Sensitivity analyses include 2015, exclude promoted content or the top 1% of accounts, use unique accounts as the outcome, and fit a piecewise slope with a knot at 2022.

In [ ]:
den = (pd.read_csv(DEN_P).dropna(subset=["total"]).groupby(["subreddit", "year"], as_index=False)["total"].sum()
         .rename(columns={"total": "exposure"}))
subs_ok = den.subreddit.unique().tolist()
def panel(frame, outcome="docs"):
    f = frame[frame.subreddit.isin(subs_ok) & frame.year.between(Y0, Y1)]
    if outcome == "accounts":
        f = f[f.author.fillna("[deleted]").ne("[deleted]")]; g = f.groupby(["subreddit", "year"]).author.nunique()
    else:
        g = f.groupby(["subreddit", "year"]).size()
    p = den[den.year.between(Y0, Y1)].merge(g.rename("y").reset_index(), on=["subreddit", "year"], how="left")
    p["y"] = p["y"].fillna(0).astype(int); p = p[p.exposure > 0].copy()
    p["year_c"] = p.year - 2018; p["post2022"] = np.clip(p.year - 2022, 0, None); return p
def nb_fit(p, formula="y ~ year_c + C(subreddit)", term="year_c", drop2015=True):
    q = p[p.year != 2015] if drop2015 else p
    m = smf.negativebinomial(formula, q, offset=np.log(q.exposure)).fit(disp=0, maxiter=1000)
    lo, hi = m.conf_int().loc[term]; pct = lambda v: round(100 * (np.exp(v) - 1), 1)
    return {"pct_per_yr": pct(m.params[term]), "ci95": [pct(lo), pct(hi)], "p": float(m.pvalues[term]),
            "alpha": round(float(m.params.get("alpha", np.nan)), 3), "n_obs": int(m.nobs)}
L_all = df[df.lpa]
base_p = panel(L_all[~L_all.promo])
R["growth_adjusted"] = {
    "primary (excl promoted)": nb_fit(base_p),
    "incl 2015": nb_fit(base_p, drop2015=False),
    "incl promoted": nb_fit(panel(L_all)),
    "excl top-1% accounts": nb_fit(panel(L_all[~L_all.promo & ~L_all.top1_author])),
    "unique accounts outcome": nb_fit(panel(L_all[~L_all.promo], "accounts")),
    "piecewise: slope before 2022": nb_fit(base_p, "y ~ year_c + post2022 + C(subreddit)", "year_c"),
    "piecewise: added slope after 2022": nb_fit(base_p, "y ~ year_c + post2022 + C(subreddit)", "post2022"),
}
# Offset vs no offset on the same panel, Poisson with robust (sandwich) SEs (stable where the NB no-offset fit fails)
import statsmodels.api as sm
_q = base_p[base_p.year != 2015]
def pois_rob(offset):
    m = smf.glm("y ~ year_c + C(subreddit)", _q, family=sm.families.Poisson(),
                offset=(np.log(_q.exposure) if offset else None)).fit(cov_type="HC0")
    lo, hi = m.conf_int().loc["year_c"]; pct = lambda v: round(100 * (np.exp(v) - 1), 1)
    return {"pct_per_yr": pct(m.params["year_c"]), "ci95": [pct(lo), pct(hi)]}
R["growth_adjusted"]["Poisson robust SE, with offset"] = pois_rob(True)
R["growth_adjusted"]["Poisson robust SE, no offset (same panel)"] = pois_rob(False)
base_p.to_csv(f"{REV}/growth_panel.csv", index=False)
for k, v in R["growth_adjusted"].items(): print(f"{k:38s} {v}")

## 13. Full-corpus sensitivity run (optional; R1.1)
Seeded refit on the full curated corpus (keyword representations only) for comparison with the original analysis.

In [ ]:
if RUN_FULL_CORPUS:
    full_modeler = TopicModeler(enable_llm=False, seed=config.SEED)
    full_modeler.fit_transform(reddit_df)
    full_modeler.reduce_outliers(strategy=config.OUTLIER_STRATEGY, threshold=config.OUTLIER_THRESHOLD)
    full_modeler.save_complete_state(f"{REV}/metadata_full_seeded.pkl", f"{REV}/model_full_seeded.bertopic")
    ft = np.array(full_modeler.topics)
    R["full_corpus_seeded"] = {"n": int(len(ft)), "n_topics": int(len(set(ft)) - (-1 in ft)),
                               "outlier_pct": round(100 * float((ft == -1).mean()), 1)}
    print(R["full_corpus_seeded"])

## 14. Export results
Writes `R` to `revision_results.json`.

In [ ]:
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o
json.dump(_clean(R), open(f"{REV}/revision_results.json", "w"), indent=2)
print("saved", f"{REV}/revision_results.json")
print(json.dumps(_clean({k: R[k] for k in ["corpus", "model", "coherence"] if k in R}), indent=1)[:3000])